# QTRL - Demonstration Notebook
This notebook demonstrates the complete workflow:
1. Initialization of the quantum-classical hybrid model
2. Weight generation via quantum circuit forward pass
3. Environment setup and data loading (CartPole-v1)
4. Policy inference using generated weights

In [ ]:
import torch
import gymnasium as gym
from lib.util import HybridMLPModel, rl_agent_forward, MinigridImageOnlyWrapper

## Step 1: Initialize the Hybrid Model
We instantiate a HybridMLPModel configured for CartPole.
- State dimension: 4
- Action dimension: 2
- Required weights: 4 × 2 = 8

In [ ]:
model = HybridMLPModel(
    q_output_size=4,
    nb_photons=2,
    nb_modes=2,
    hidden_sizes=[16, 16],
    final_output_size=8
)
print('QTRL HybridMLPModel successfully initialized.')

## Step 2: Generate Weights via Quantum Circuit
The forward pass of the model combines the quantum photonic circuit with classical mapping to produce policy weights.

In [ ]:
with torch.no_grad():
    generated_weights = model()[0]
    
print(f'Shape of generated weights: {generated_weights.shape}')
print(f'First 8 weights (policy parameters): {generated_weights[:8]}')
print(f'Contains NaN: {torch.isnan(generated_weights).any().item()}')

## Step 3: Load Environment and Prepare Observation
We initialize the CartPole environment and obtain the initial observation state.

In [ ]:
env = gym.make('CartPole-v1')
obs, info = env.reset(seed=42)

print(f'Environment: CartPole-v1')
print(f'Initial observation shape: {obs.shape}')
print(f'Observation: {obs}')

## Step 4: Policy Inference
We compute action logits using the quantum-generated weights and the current observation.

In [ ]:
# Convert observation to tensor
obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)

# Compute policy logits using generated weights
logits = rl_agent_forward(
    obs_tensor, 
    generated_weights[:8], 
    input_dim=4, 
    output_dim=2
)

# Select action via argmax
action = torch.argmax(logits).item()

print(f'Observation tensor shape: {obs_tensor.shape}')
print(f'Policy logits: {logits}')
print(f'Selected action: {action}')

## Step 5: Execute Step in Environment
We apply the selected action and observe the environment response.

In [ ]:
next_obs, reward, terminated, truncated, info = env.step(action)

print(f'Action taken: {action}')
print(f'Reward received: {reward}')
print(f'Episode terminated: {terminated}')
print(f'Episode truncated: {truncated}')
print(f'Next observation: {next_obs}')

env.close()
print('\nEnvironment closed. Demonstration complete.')